In [ ]:
import os 
os.getcwd()

In [ ]:
import json
import pandas as pd
from pathlib import Path
from collections import Counter

DATA_ROOT = Path("./")
AOKVQA_DIR = DATA_ROOT / "aokvqa"
COCO_DIR = DATA_ROOT / "coco"

JSON_FILES = {
    "train": AOKVQA_DIR / "aokvqa_v1p0_train.json",
    "val": AOKVQA_DIR / "aokvqa_v1p0_val.json", # important use val as test with ground truth available
    "test": AOKVQA_DIR / "aokvqa_v1p0_test.json",
}

def choose_answer(item):
    if "choices" in item and "correct_choice_idx" in item:
        idx = item.get("correct_choice_idx")
        if isinstance(idx, int) and 0 <= idx < len(item["choices"]):
            return item["choices"][idx]
    das = item.get("direct_answers") or []
    das = [s.lower().strip() for s in das if s]
    if not das:
        return ""
    counts = Counter(das)
    return counts.most_common(1)[0][0]

def choose_rationale(item):
    rats = item.get("rationales") or []
    return rats[0] if rats else ""

def format_choices(item):
    opts = item.get("choices") or []
    return "; ".join(f"{opt.lower().strip()}" for i, opt in enumerate(opts))


def format_idx_choices(item):
    opts = item.get("choices") or []
    labels = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    return "\n".join(f"({labels[i]}) {opt.lower().strip()}" for i, opt in enumerate(opts))
    # return "; ".join(f"{opt.lower().strip()}" for i, opt in enumerate(opts))


def coco_image_path(split, image_id):
    split_dir = f"{split}2017"
    filename = f"{image_id:012d}.jpg"
    return str(COCO_DIR / split_dir / filename)


def map_image_path(p):
    s = str(p)
    s = s.replace("coco/", "data/images/aokvqa/")
    return s
    
def build_df(split, json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        items = json.load(f)
    rows = []
    for it in items:
        rows.append({
            "image_path": coco_image_path(split, it["image_id"]),
            "question": it.get("question", ""),
            "answer": choose_answer(it),
            "rationale": choose_rationale(it),
            "choices": format_choices(it),
            "idx_choices": format_idx_choices(it),
        })
    df = pd.DataFrame(rows, columns=["image_path", "question", "answer", "rationale", "choices", "idx_choices"])
    df["image_path"] = df["image_path"].apply(map_image_path)
    return df



In [ ]:

df_train = build_df("train", JSON_FILES["train"])
df_val = build_df("val", JSON_FILES["val"])
df_test = build_df("test", JSON_FILES["test"])
df_train

cleaning

In [ ]:
import re
def extract_choice_pairs(s: str):
    """Order-agnostic parse of lines like '(A) foo', '(B) bar', ...
    Returns list of (letter, text) in the order they appear.
    """
    pairs = re.findall(r"\(([A-D])\)\s*(.+)", s)
    return [(ltr, txt.strip()) for (ltr, txt) in pairs]


def extract_choices(question: str):
    """Order-agnostic: return only the option texts in the order they appear."""
    return [txt for (_ltr, txt) in extract_choice_pairs(question)]

# uses your extract_choice_pairs()

def add_label_letter_df(df):
    out = df.copy()
    label_letters = []
    for _, r in out.iterrows():
        pairs = extract_choice_pairs(r["idx_choices"])
        letter = None
        ans = str(r["answer"]).strip().lower()
        for ltr, txt in pairs:
            if str(txt).strip().lower() == ans:
                letter = ltr
                break
        label_letters.append(letter)
    out["label_letter"] = label_letters
    return out

def find_bad_rows(df):
    bad_idx = []
    for i, r in df.iterrows():
        pairs = extract_choice_pairs(r["idx_choices"])
        # bad if not exactly 4 parsed options or no matching letter found
        if len(pairs) != 4 or pd.isna(r["label_letter"]):
            bad_idx.append(i)
    return df.loc[bad_idx]

def drop_bad_rows(df: pd.DataFrame) -> pd.DataFrame:
    """Return a DataFrame with the same columns, dropping rows where
    - choices don't parse to exactly 4 options, or
    - answer doesn't match any option (no label_letter).
    """
    ann = add_label_letter_df(df)
    bad = find_bad_rows(ann)
    keep_idx = ann.index.difference(bad.index)
    return df.loc[keep_idx].copy()



In [ ]:
df_train = drop_bad_rows(df_train)
df_val = drop_bad_rows(df_val)
df_test = drop_bad_rows(df_test)
df_train

In [ ]:
print(len(df_train))
print(df_train.head())
print(len(df_val))
print(df_val.head())
print(len(df_test))
print(df_test.head())

In [ ]:
# Assign globally unique IDs across FVQA splits before writing Parquet files
splits = [
    ("train", df_train),
    ("val", df_val),
    ("test", df_test),
]

combined = pd.concat(
    [df.reset_index(drop=True).assign(__split=name) for name, df in splits],
    ignore_index=True,
)

combined["uid"] = range(1, len(combined) + 1)

def _restore_split(name):
    df_split = (
        combined.loc[combined["__split"] == name]
        .drop(columns="__split")
        .reset_index(drop=True)
    )
    cols = ["uid"] + [c for c in df_split.columns if c != "uid"]
    return df_split.loc[:, cols]

df_train = _restore_split("train")
df_val = _restore_split("val")
df_test = _restore_split("test")

In [ ]:
print(len(df_train))
print(df_train.head())
print(len(df_val))
print(df_val.head())
print(len(df_test))
print(df_test.head())

In [ ]:
PARQUET_DIR = AOKVQA_DIR / "parquet"
PARQUET_DIR.mkdir(parents=True, exist_ok=True)

df_train.to_parquet(PARQUET_DIR / "train.parquet", index=False)
df_test.to_parquet(PARQUET_DIR / "val.parquet", index=False)
df_val.to_parquet(PARQUET_DIR / "test.parquet", index=False)

In [ ]:
from tokens import HF_TOKEN
from huggingface_hub import HfApi, create_repo, upload_folder, upload_file
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()

api = HfApi(token=HF_TOKEN)
repo_id = "JJoy333/RationaleVQA"
create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)


upload_folder(
    folder_path=str(PARQUET_DIR),
    repo_id=repo_id,
    repo_type="dataset",
    path_in_repo="AOKVQA"
)
